# 05 — Compare Triton with CUDA C++

**Goal:** implement the same vector-add operation using a small CUDA C++ extension and compare it with the Triton version.

This notebook is optional because building a CUDA extension requires a compatible CUDA compiler (`nvcc`) and matching PyTorch environment. If the compiler is missing, the notebook explains what to check and the other experiments remain usable.

## 1. Check the runtime and compiler

In [ ]:
import shutil
import torch
import triton
import triton.language as tl

print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("CUDA available to PyTorch:", torch.cuda.is_available())
print("nvcc path:", shutil.which("nvcc"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Define a small CUDA extension

CUDA C++ explicitly writes a kernel and declares how many threads are in each CUDA block. Here, each thread calculates one array element. The C++ wrapper checks the inputs and launches the GPU kernel on PyTorch's current CUDA stream.

The code is kept here rather than hidden in a downloaded binary so you can read it, modify it, and understand the comparison.

In [ ]:
cpp_source = r"""
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>

void vector_add_cuda(torch::Tensor x, torch::Tensor y, torch::Tensor out);

void vector_add_(torch::Tensor x, torch::Tensor y, torch::Tensor out) {
    TORCH_CHECK(x.is_cuda() && y.is_cuda() && out.is_cuda(), "Tensors must be on CUDA");
    TORCH_CHECK(x.scalar_type() == torch::kFloat32, "x must be float32");
    TORCH_CHECK(y.scalar_type() == torch::kFloat32, "y must be float32");
    TORCH_CHECK(out.scalar_type() == torch::kFloat32, "out must be float32");
    TORCH_CHECK(x.is_contiguous() && y.is_contiguous() && out.is_contiguous(),
                "Tensors must be contiguous");
    TORCH_CHECK(x.sizes() == y.sizes() && x.sizes() == out.sizes(),
                "Tensor shapes must match");
    vector_add_cuda(x, y, out);
}
"""

cuda_source = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void vector_add_kernel(const float* x, const float* y,
                                 float* out, long long n) {
    long long i = static_cast<long long>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (i < n) {
        out[i] = x[i] + y[i];
    }
}

void vector_add_cuda(torch::Tensor x, torch::Tensor y, torch::Tensor out) {
    long long n = x.numel();
    if (n == 0) return;
    int threads = 256;
    int blocks = static_cast<int>((n + threads - 1) / threads);
    vector_add_kernel<<<blocks, threads, 0, at::cuda::getCurrentCUDAStream()>>>(
        x.data_ptr<float>(), y.data_ptr<float>(), out.data_ptr<float>(), n);
    C10_CUDA_KERNEL_LAUNCH_CHECK();
}
"""

## 3. Build the extension when supported

Compilation may take a few minutes the first time. Do not install a random CUDA toolkit into a working Colab runtime if the versions are incompatible. If compilation fails, save the error message and investigate it rather than silently changing the environment.

In [ ]:
from torch.utils.cpp_extension import load_inline

cuda_add_module = None
if not torch.cuda.is_available():
    print("Skipped: select a GPU runtime first.")
elif shutil.which("nvcc") is None:
    print("Skipped: nvcc was not found in this runtime.")
else:
    try:
        cuda_add_module = load_inline(
            name="triton_lab_cuda_add_extension",
            cpp_sources=cpp_source,
            cuda_sources=cuda_source,
            functions=["vector_add_"],
            verbose=True,
        )
        print("CUDA C++ extension compiled successfully")
    except Exception as exc:
        print("Extension build failed. Save the full traceback before changing packages.")
        raise

## 4. Add a Triton reference in the same notebook

This keeps the operation constant while changing the way the GPU kernel is expressed.

In [ ]:
@triton.jit
def vector_add_kernel(x_ptr, y_ptr, out_ptr, n_elements: tl.constexpr,
                      BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def launch_triton_add(x, y, out, block_size=256):
    grid = (triton.cdiv(x.numel(), block_size),)
    vector_add_kernel[grid](x, y, out, x.numel(), BLOCK_SIZE=block_size)

## 5. Compare correctness and timing

In [ ]:
import statistics

def median_gpu_time_ms(fn, warmup=10, repeats=50, rounds=5):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(rounds):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(repeats):
            fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end) / repeats)
    return statistics.median(samples), samples

if cuda_add_module is not None:
    n = 1 << 20
    x = torch.randn(n, device="cuda", dtype=torch.float32)
    y = torch.randn_like(x)
    out_torch = torch.empty_like(x)
    out_triton = torch.empty_like(x)
    out_cuda = torch.empty_like(x)

    def run_torch():
        torch.add(x, y, out=out_torch)

    def run_triton():
        launch_triton_add(x, y, out_triton, block_size=256)

    def run_cuda_cpp():
        cuda_add_module.vector_add_(x, y, out_cuda)

    run_torch(); run_triton(); run_cuda_cpp()
    torch.testing.assert_close(out_triton, out_torch)
    torch.testing.assert_close(out_cuda, out_torch)

    for label, fn in [("PyTorch", run_torch), ("Triton", run_triton), ("CUDA C++", run_cuda_cpp)]:
        ms, rounds = median_gpu_time_ms(fn)
        print(f"{label:10s}: {ms:.6f} ms/call; rounds={['%.6f' % v for v in rounds]}")
else:
    print("Timing skipped because the CUDA extension was not built.")

## Final notes

Triton often makes custom GPU kernels easier to write. CUDA C++ gives more direct control but requires more explicit kernel and build details. A single vector-add timing result does not decide which approach is better overall. Record compiler/toolkit versions, correctness, build issues, and measured timing. The next research projects—FlexKV and PagedKV—should begin only after these kernel basics are understood.